In [13]:
print("hello world ")

hello world 


In [14]:
%pwd

'c:\\Users\\vivobook\\Desktop\\INPT\\Me\\Project\\developpementProject\\chatBotJuridiques-RAG-'

In [15]:
import os 
os.chdir("../data")
%pwd

'c:\\Users\\vivobook\\Desktop\\INPT\\Me\\Project\\developpementProject\\data'

In [ ]:
%pwd

'c:\\Users\\vivobook\\Desktop\\INPT\\Me\\Project\\developpementProject\\data'

In [17]:
import os 
os.chdir("../")


In [18]:
%pwd

'c:\\Users\\vivobook\\Desktop\\INPT\\Me\\Project\\developpementProject'

In [19]:
from langchain.document_loaders import PyPDFLoader , DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
def loadPdfFilles(path):
    loader=DirectoryLoader(
        path,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents=loader.load()
    return documents

In [20]:
extractedData=loadPdfFilles("data")


In [21]:
extractedData # list of pages 

[]

In [22]:
len(extractedData) #nbr of pages

0

In [ ]:
from pathlib import Path
from pdf2image import convert_from_path
from PIL import Image
import pytesseract
import json
import os

current_path = Path.cwd()
BASE_DIR = current_path.parent  # Set BASE_DIR to parent directory where data folder is located

PDF_DIR = BASE_DIR / "data"
OUT_DIR = BASE_DIR / "data" / "pages"
OCR_OUT_DIR = BASE_DIR / "artifacts" / "ocr"

DPI = 150
POPPLER_PATH = r"C:\poppler\Library\bin"
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
SLICE_HEIGHT = 80 

def extract_patches(img, slice_height=SLICE_HEIGHT):
    w, h = img.size
    patches = []
    coords = []
    for top in range(0, h, slice_height):
        box = (0, top, w, min(top + slice_height, h))
        patch = img.crop(box)
        patches.append(patch)
        coords.append(box)
    return patches, coords

def ocr_patches(patches):
    texts = []
    for patch in patches:
        text = pytesseract.image_to_string(patch, lang='ara', config='--psm 6')
        texts.append(text.strip())
    return texts

def process_all_pdfs():
    print(f"--- Working Directory: {BASE_DIR} ---")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    OCR_OUT_DIR.mkdir(parents=True, exist_ok=True)
    
    pdfs = list(PDF_DIR.glob("*.pdf"))
    if not pdfs:
        print(f"❌ Error: No PDFs found in {PDF_DIR}")
        return

    print(f"--- STEP 1: CONVERTING {len(pdfs)} PDFs TO IMAGES ---")
    for pdf_path in pdfs:
        doc_id = pdf_path.stem
        doc_out = OUT_DIR / doc_id
        doc_out.mkdir(parents=True, exist_ok=True)
        try:
            pages = convert_from_path(pdf_path, dpi=DPI, fmt="png", poppler_path=POPPLER_PATH)
            for i, page in enumerate(pages, start=1):
                page_path = doc_out / f"page_{i:03d}.png"
                if not page_path.exists():
                    page.save(page_path, "PNG")
            print(f"✅ {doc_id}: {len(pages)} pages processed.")
        except Exception as e:
            print(f"❌ Error converting {pdf_path.name}: {e}")

    print("\n--- STEP 2: EXTRACTING TEXT (OCR) ---")
    for doc_folder in OUT_DIR.iterdir():
        if not doc_folder.is_dir(): continue
        
        pages_images = sorted(doc_folder.glob("*.png"))
        print(f"Processing: {doc_folder.name}")

        for img_path in pages_images:
            img = Image.open(img_path).convert("RGB")
            patches, coords = extract_patches(img)
            ocr_texts = ocr_patches(patches)

            ocr_data = {
                "source_pdf": doc_folder.name,
                "page_image": img_path.name,
                "data": []
            }
            
            for i, (bbox, text) in enumerate(zip(coords, ocr_texts)):
                if text and len(text) > 2: 
                    ocr_data["data"].append({
                        "patch_index": i,
                        "bbox": bbox,
                        "text": text
                    })

            json_name = f"{doc_folder.name}_{img_path.stem}_ocr.json"
            with open(OCR_OUT_DIR / json_name, "w", encoding="utf-8") as f:
                json.dump(ocr_data, f, ensure_ascii=False, indent=4)

    print("\n🎉 All tasks completed successfully!")

if __name__ == "__main__":
    process_all_pdfs()

--- Working Directory: c:\Users\vivobook\Desktop\INPT\Me\Project ---
❌ Error: No PDFs found in c:\Users\vivobook\Desktop\INPT\Me\Project\data


In [ ]:
from pathlib import Path
from pdf2image import convert_from_path
from PIL import Image
import pytesseract
import json
import os

# --- FIX 1: Smart Pathing ---
# This checks if "data" is in the current folder, or the parent folder
if (Path.cwd() / "data").exists():
    BASE_DIR = Path.cwd()
else:
    BASE_DIR = Path.cwd().parent

PDF_DIR = BASE_DIR / "data"
OUT_DIR = BASE_DIR / "data" / "pages"
OCR_OUT_DIR = BASE_DIR / "artifacts" / "ocr"

DPI = 150
POPPLER_PATH = r"C:\poppler\Library\bin"
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
SLICE_HEIGHT = 80 

def extract_patches(img, slice_height=SLICE_HEIGHT):
    w, h = img.size
    patches = []
    coords = []
    for top in range(0, h, slice_height):
        box = (0, top, w, min(top + slice_height, h))
        patch = img.crop(box)
        patches.append(patch)
        coords.append(box)
    return patches, coords

def ocr_patches(patches):
    texts = []
    total = len(patches)
    for i, patch in enumerate(patches):
        # --- FIX 2: Added progress print for patches ---
        print(f"      -> Extracting text from patch {i+1}/{total}...", end="\r")
        text = pytesseract.image_to_string(patch, lang='ara', config='--psm 6')
        texts.append(text.strip())
    print() # Go to next line after patches are done
    return texts

def process_all_pdfs():
    print(f"--- Working Directory: {BASE_DIR} ---")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    OCR_OUT_DIR.mkdir(parents=True, exist_ok=True)
    
    pdfs = list(PDF_DIR.glob("*.pdf"))
    if not pdfs:
        print(f"❌ Error: No PDFs found in {PDF_DIR}")
        return

    print(f"--- STEP 1: CONVERTING {len(pdfs)} PDFs TO IMAGES ---")
    for pdf_path in pdfs:
        doc_id = pdf_path.stem
        doc_out = OUT_DIR / doc_id
        doc_out.mkdir(parents=True, exist_ok=True)
        try:
            print(f"⏳ Converting {pdf_path.name}...")
            pages = convert_from_path(pdf_path, dpi=DPI, fmt="png", poppler_path=POPPLER_PATH)
            for i, page in enumerate(pages, start=1):
                page_path = doc_out / f"page_{i:03d}.png"
                if not page_path.exists():
                    page.save(page_path, "PNG")
            print(f"✅ {doc_id}: {len(pages)} pages processed.")
        except Exception as e:
            print(f"❌ Error converting {pdf_path.name}: {e}")

    print("\n--- STEP 2: EXTRACTING TEXT (OCR) ---")
    for doc_folder in OUT_DIR.iterdir():
        if not doc_folder.is_dir(): continue
        
        pages_images = sorted(doc_folder.glob("*.png"))
        print(f"\n📂 Processing Document: {doc_folder.name}")

        for img_path in pages_images:
            print(f"  📄 Reading {img_path.name}...")
            img = Image.open(img_path).convert("RGB")
            patches, coords = extract_patches(img)
            
            ocr_texts = ocr_patches(patches)

            ocr_data = {
                "source_pdf": doc_folder.name,
                "page_image": img_path.name,
                "data": []
            }
            
            for i, (bbox, text) in enumerate(zip(coords, ocr_texts)):
                if text and len(text) > 2: 
                    ocr_data["data"].append({
                        "patch_index": i,
                        "bbox": bbox,
                        "text": text
                    })

            json_name = f"{doc_folder.name}_{img_path.stem}_ocr.json"
            with open(OCR_OUT_DIR / json_name, "w", encoding="utf-8") as f:
                json.dump(ocr_data, f, ensure_ascii=False, indent=4)
            print(f"  💾 Saved JSON for {img_path.name}")

    print("\n🎉 All tasks completed successfully!")

if __name__ == "__main__":
    process_all_pdfs()

In [ ]:
%pwd

'c:\\Users\\vivobook\\Desktop\\INPT\\Me\\Project\\developpementProject'

In [ ]:
from pathlib import Path
from pdf2image import convert_from_path
from PIL import Image
import pytesseract
import json
import os

# ==========================================
# 1. BULLETPROOF PATHING
# ==========================================
current_path = Path.cwd()

# If running from inside the 'research' folder, go up one level
if current_path.name == "research":
    BASE_DIR = current_path.parent 
else:
    # Failsafe: Hardcoded absolute path
    BASE_DIR = Path(r"C:\Users\vivobook\Desktop\INPT\Me\Project\developpementProject\chatBotJuridiques-RAG-")

PDF_DIR = BASE_DIR / "data"
OUT_DIR = BASE_DIR / "data" / "pages"
OCR_OUT_DIR = BASE_DIR / "artifacts" / "ocr"

# ==========================================
# 2. CONFIGURATION
# ==========================================
DPI = 150
POPPLER_PATH = r"C:\poppler\Library\bin"
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
SLICE_HEIGHT = 80 

def extract_patches(img, slice_height=SLICE_HEIGHT):
    w, h = img.size
    patches = []
    coords = []
    for top in range(0, h, slice_height):
        box = (0, top, w, min(top + slice_height, h))
        patch = img.crop(box)
        patches.append(patch)
        coords.append(box)
    return patches, coords

def ocr_patches(patches):
    texts = []
    total = len(patches)
    for i, patch in enumerate(patches):
        # Progress indicator so you know it's not frozen
        print(f"      -> Extracting text from patch {i+1}/{total}...", end="\r")
        text = pytesseract.image_to_string(patch, lang='ara', config='--psm 6')
        texts.append(text.strip())
    print() # Move to the next line after finishing the patches
    return texts

def process_all_pdfs():
    print(f"--- Working Directory: {BASE_DIR} ---")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    OCR_OUT_DIR.mkdir(parents=True, exist_ok=True)
    
    pdfs = list(PDF_DIR.glob("*.pdf"))
    if not pdfs:
        print(f"❌ Error: No PDFs found in {PDF_DIR}")
        return

    print(f"\n--- STEP 1: CONVERTING {len(pdfs)} PDFs TO IMAGES ---")
    for pdf_path in pdfs:
        doc_id = pdf_path.stem
        doc_out = OUT_DIR / doc_id
        doc_out.mkdir(parents=True, exist_ok=True)
        try:
            print(f"⏳ Converting {pdf_path.name}...")
            pages = convert_from_path(pdf_path, dpi=DPI, fmt="png", poppler_path=POPPLER_PATH)
            for i, page in enumerate(pages, start=1):
                page_path = doc_out / f"page_{i:03d}.png"
                if not page_path.exists():
                    page.save(page_path, "PNG")
            print(f"✅ {doc_id}: {len(pages)} pages processed.")
        except Exception as e:
            print(f"❌ Error converting {pdf_path.name}: {e}")

    print("\n--- STEP 2: EXTRACTING TEXT (OCR) ---")
    for doc_folder in OUT_DIR.iterdir():
        if not doc_folder.is_dir(): continue
        
        pages_images = sorted(doc_folder.glob("*.png"))
        print(f"\n📂 Processing Document: {doc_folder.name}")

        for img_path in pages_images:
            print(f"  📄 Reading {img_path.name}...")
            img = Image.open(img_path).convert("RGB")
            patches, coords = extract_patches(img)
            
            ocr_texts = ocr_patches(patches)

            ocr_data = {
                "source_pdf": doc_folder.name,
                "page_image": img_path.name,
                "data": []
            }
            
            for i, (bbox, text) in enumerate(zip(coords, ocr_texts)):
                if text and len(text) > 2: 
                    ocr_data["data"].append({
                        "patch_index": i,
                        "bbox": bbox,
                        "text": text
                    })

            json_name = f"{doc_folder.name}_{img_path.stem}_ocr.json"
            with open(OCR_OUT_DIR / json_name, "w", encoding="utf-8") as f:
                json.dump(ocr_data, f, ensure_ascii=False, indent=4)
            print(f"  💾 Saved JSON for {img_path.name}")

    print("\n🎉 All tasks completed successfully!")

process_all_pdfs()

In [1]:
from pathlib import Path
from pdf2image import convert_from_path
from PIL import Image
import pytesseract
import json
import os

# ==========================================
# 1. BULLETPROOF PATHING
# ==========================================
current_path = Path.cwd()

# If running from inside the 'research' folder, go up one level
if current_path.name == "research":
    BASE_DIR = current_path.parent 
else:
    # Failsafe: Hardcoded absolute path
    BASE_DIR = Path(r"C:\Users\vivobook\Desktop\INPT\Me\Project\developpementProject\chatBotJuridiques-RAG-")

PDF_DIR = BASE_DIR / "data"
OUT_DIR = BASE_DIR / "data" / "pages"
OCR_OUT_DIR = BASE_DIR / "artifacts" / "ocr"

# ==========================================
# 2. CONFIGURATION
# ==========================================
DPI = 150
POPPLER_PATH = r"C:\poppler\Library\bin"
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
SLICE_HEIGHT = 80 

def extract_patches(img, slice_height=SLICE_HEIGHT):
    w, h = img.size
    patches = []
    coords = []
    for top in range(0, h, slice_height):
        box = (0, top, w, min(top + slice_height, h))
        patch = img.crop(box)
        patches.append(patch)
        coords.append(box)
    return patches, coords

def ocr_patches(patches):
    texts = []
    total = len(patches)
    for i, patch in enumerate(patches):
        # Progress indicator so you know it's not frozen
        print(f"      -> Extracting text from patch {i+1}/{total}...", end="\r")
        text = pytesseract.image_to_string(patch, lang='ara', config='--psm 6')
        texts.append(text.strip())
    print() # Move to the next line after finishing the patches
    return texts

def process_all_pdfs():
    print(f"--- Working Directory: {BASE_DIR} ---")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    OCR_OUT_DIR.mkdir(parents=True, exist_ok=True)
    
    pdfs = list(PDF_DIR.glob("*.pdf"))
    if not pdfs:
        print(f"❌ Error: No PDFs found in {PDF_DIR}")
        return

    print(f"\n--- STEP 1: CONVERTING {len(pdfs)} PDFs TO IMAGES ---")
    for pdf_path in pdfs:
        doc_id = pdf_path.stem
        doc_out = OUT_DIR / doc_id
        doc_out.mkdir(parents=True, exist_ok=True)
        try:
            print(f"⏳ Converting {pdf_path.name}...")
            pages = convert_from_path(pdf_path, dpi=DPI, fmt="png", poppler_path=POPPLER_PATH)
            for i, page in enumerate(pages, start=1):
                page_path = doc_out / f"page_{i:03d}.png"
                if not page_path.exists():
                    page.save(page_path, "PNG")
            print(f"✅ {doc_id}: {len(pages)} pages processed.")
        except Exception as e:
            print(f"❌ Error converting {pdf_path.name}: {e}")

    print("\n--- STEP 2: EXTRACTING TEXT (OCR) ---")
    for doc_folder in OUT_DIR.iterdir():
        if not doc_folder.is_dir(): continue
        
        pages_images = sorted(doc_folder.glob("*.png"))
        print(f"\n📂 Processing Document: {doc_folder.name}")

        for img_path in pages_images:
            print(f"  📄 Reading {img_path.name}...")
            img = Image.open(img_path).convert("RGB")
            patches, coords = extract_patches(img)
            
            ocr_texts = ocr_patches(patches)

            ocr_data = {
                "source_pdf": doc_folder.name,
                "page_image": img_path.name,
                "data": []
            }
            
            for i, (bbox, text) in enumerate(zip(coords, ocr_texts)):
                if text and len(text) > 2: 
                    ocr_data["data"].append({
                        "patch_index": i,
                        "bbox": bbox,
                        "text": text
                    })

            json_name = f"{doc_folder.name}_{img_path.stem}_ocr.json"
            with open(OCR_OUT_DIR / json_name, "w", encoding="utf-8") as f:
                json.dump(ocr_data, f, ensure_ascii=False, indent=4)
            print(f"  💾 Saved JSON for {img_path.name}")

    print("\n🎉 All tasks completed successfully!")

process_all_pdfs()

--- Working Directory: c:\Users\vivobook\Desktop\INPT\Me\Project\developpementProject\chatBotJuridiques-RAG- ---

--- STEP 1: CONVERTING 13 PDFs TO IMAGES ---
⏳ Converting AMTJ-parents.pdf...
✅ AMTJ-parents: 2 pages processed.
⏳ Converting cir 01-retraite complémentaire 1.pdf...
✅ cir 01-retraite complémentaire 1: 1 pages processed.
⏳ Converting cir 2-Pèlerinage.pdf...
✅ cir 2-Pèlerinage: 1 pages processed.
⏳ Converting CIR 7-transport aérien.pdf...
✅ CIR 7-transport aérien: 1 pages processed.
⏳ Converting CIR 8-convention transport.pdf...
✅ CIR 8-convention transport: 1 pages processed.
⏳ Converting Cir don femmes enceintes26.pdf...
✅ Cir don femmes enceintes26: 1 pages processed.
⏳ Converting com 06-convention g_Omrane.pdf...
✅ com 06-convention g_Omrane: 1 pages processed.
⏳ Converting com 4 AMC.pdf...
✅ com 4 AMC: 10 pages processed.
⏳ Converting COM 5-séjour linguis.pdf...
✅ COM 5-séjour linguis: 1 pages processed.
⏳ Converting com convention Eqdom.pdf...
✅ com convention 

In [2]:
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return a Multilingual embedding model optimized for Arabic.
    """
    model_name = "intfloat/multilingual-e5-base"
    
    model_kwargs = {'device': 'cpu'} 
    encode_kwargs = {'normalize_embeddings': True} 
    
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs
    )
    return embeddings

embedding = download_embeddings()
print("✅ Embeddings model is ready for Arabic!")

ImportError: cannot import name 'cookiejar' from 'http' (c:\Users\vivobook\miniconda3\envs\medibot\lib\http\__init__.py)

In [ ]:
vector = embedding.embed_query("الإقتصاد")
len(vector)

768

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_core.documents import Document
from langchain_pinecone import PineconeVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings

load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
index_name = "juridique-rag-index"

pc = Pinecone(api_key=PINECONE_API_KEY)

print("✅ Setup is ready! Connected to Pinecone.")

c:\Users\vivobook\miniconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Setup is ready! Connected to Pinecone.


In [ ]:
print(" Loading multilingual-e5-base model...")

embedding = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={'device': 'cpu'}, 
    encode_kwargs={'normalize_embeddings': True}
)

print("✅ Model loaded successfully!")

 Loading multilingual-e5-base model...


C:\Users\vivobook\AppData\Local\Temp\ipykernel_9408\3800351749.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


✅ Model loaded successfully!


In [ ]:
if not pc.has_index(index_name):
    print(f"⏳ Creating a new index called '{index_name}'...")
    pc.create_index(
        name=index_name,
        dimension=768,  
        metric="cosine", 
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print("✅ Index created successfully!")
else:
    print(f"✅ Index '{index_name}' already exists. We will use it.")

✅ Index 'juridique-rag-index' already exists. We will use it.


In [ ]:
%pwd

'c:\\Users\\vivobook\\Desktop\\INPT\\Me\\Project\\developpementProject\\chatBotJuridiques-RAG-\\research'

In [ ]:
current_path = Path.cwd()
BASE_DIR = current_path.parent if current_path.name == "research" else current_path
OCR_DIR = BASE_DIR / "artifacts" / "ocr"

json_files = list(OCR_DIR.glob("*.json"))
print(f"📂 Found {len(json_files)} JSON files. Reading them...")

documents = []

for file in json_files:
    with open(file, 'r', encoding='utf-8') as f:
        content = json.load(f)
        
        for item in content.get('data', []):
            text = item.get('text', '').strip()
            
            if text and len(text) > 2:
                e5_text = f"passage: {text}"
                
                metadata = {
                    "source": content.get('source_pdf', 'Unknown'),
                    "page": content.get('page_image', 'Unknown')
                }
                
                doc = Document(page_content=e5_text, metadata=metadata)
                documents.append(doc)

print(f" Prepared {len(documents)} chunks (paragraphs) ready for Pinecone.")

📂 Found 24 JSON files. Reading them...
 Prepared 438 chunks (paragraphs) ready for Pinecone.


In [ ]:
print(" Uploading vectors to Pinecone... ")

docsearch = PineconeVectorStore.from_documents(
    documents=documents,
    embedding=embedding,
    index_name=index_name
)

print(" DONE! All your legal documents are now vectorized and stored in Pinecone!")

 Uploading vectors to Pinecone... 
 DONE! All your legal documents are now vectorized and stored in Pinecone!


In [ ]:
from langchain_core.documents import Document

dswith = Document(
    page_content="passage: سلام",
    metadata={"source": "me", "page": "none"} 
)

print("⏳ Adding manual document to Pinecone...")
print(docsearch.add_documents(documents=[dswith]))
print("✅ Document added successfully!")

⏳ Adding manual document to Pinecone...
['9b88335a-8523-4a43-904a-13f31fcda9e3']
✅ Document added successfully!


In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [ ]:
retrieved_docs = retriever.invoke("اتفاقيات مع أطباء")
retrieved_docs

[Document(id='f3d22d26-f0b6-4bae-a55a-73bc64f23095', metadata={'page': 'page_005.png', 'source': 'com 4 AMC'}, page_content='passage: الاتفاقيات مع مختبرات التحليلات الطبية ,'),
 Document(id='25e568a5-98d2-4159-9482-65838ecaecff', metadata={'page': 'page_005.png', 'source': 'com 4 AMC'}, page_content='passage: الاتفاقيات مع مختبرات التحليلات الطبية ,'),
 Document(id='022b4b4a-f4c8-48dc-8e08-c4474d949b5b', metadata={'page': 'page_007.png', 'source': 'com 4 AMC'}, page_content='passage: اتفاقيات مع أطباء في عدة تخصصات (طب الأسنان - طب النساء والتوليد... إلخ) |')]

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

# Make sure to add GOOGLE_API_KEY to your .env file
chatModel = ChatGoogleGenerativeAI(model="gemini-1.5-pro-latest")


In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an expert legal assistant specializing in Moroccan law. "
    "Use the following pieces of retrieved context to answer the user's question. "
    "If the answer is not in the provided context, just say that you don't know. "
    "DO NOT make up or hallucinate any legal information. "
    "Keep your answer clear, concise, and strictly based on the context. "
    "ALWAYS answer in Arabic, regardless of the language of the prompt."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input": "تفاقيات مع أطباء ماهي"})
print(response["answer"])

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}